# Data Preprocessing Test Notebook

This notebook tests preprocessing functions and shows clear output comparisons.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == 'tests' else Path.cwd()
src_path = project_root / 'src'
dataset_dir = project_root / 'Dataset'
train_csv = dataset_dir / 'train.csv'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from preprocessing import (
    to_lower,
    remove_punctuation_and_digits,
    remove_stopwords,
    apply_stemming,
    preprocess_text,
    preprocess_dataframe,
)

print('Project root :', project_root)
print('Train exists :', train_csv.exists())

## 1) Step-by-step preprocessing comparison on sample text

In [ ]:
sample_text = 'This is A Sample Resume text, with 123 numbers and symbols!!! Running quickly.'

step_lower = to_lower(sample_text)
step_clean = remove_punctuation_and_digits(step_lower)
step_no_stopwords = remove_stopwords(step_clean)
step_stemmed = apply_stemming(step_no_stopwords)

steps_df = pd.DataFrame({
    'Stage': ['Original', 'Lowercase', 'Cleaned', 'Stopwords Removed', 'Stemmed'],
    'Text': [sample_text, step_lower, step_clean, step_no_stopwords, step_stemmed],
})
steps_df

## 2) Validate preprocess_text output

In [ ]:
final_output = preprocess_text(sample_text)
print('preprocess_text output:', final_output)
print('Matches final staged output:', final_output == step_stemmed)

## 3) Real dataset comparison (first 5 rows)

In [ ]:
raw_df = pd.read_csv(train_csv)
text_col = 'text' if 'text' in raw_df.columns else 'Text'

source_df = raw_df[[text_col]].copy().head(5)
source_df = source_df.rename(columns={text_col: 'text'})
processed_df = preprocess_dataframe(source_df, text_column='text')

comparison_df = pd.DataFrame({
    'Original Text': source_df['text'],
    'Processed Text': processed_df['text'],
})
comparison_df

## 4) Clear numeric change comparison

In [ ]:
metrics_df = pd.DataFrame({
    'Original Length': comparison_df['Original Text'].astype(str).str.len(),
    'Processed Length': comparison_df['Processed Text'].astype(str).str.len(),
    'Original Tokens': comparison_df['Original Text'].astype(str).str.split().str.len(),
    'Processed Tokens': comparison_df['Processed Text'].astype(str).str.split().str.len(),
})
metrics_df['Length Reduction'] = metrics_df['Original Length'] - metrics_df['Processed Length']
metrics_df['Token Reduction'] = metrics_df['Original Tokens'] - metrics_df['Processed Tokens']
metrics_df